In [22]:
# Import required libraries

import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [23]:
# Load raw vendor dataset

df = pd.read_csv("../data/raw/vendors.csv")

# Display first five records
df.head()

,vendor_id,vendor_name,category,location,min_price_per_plate,max_price_per_plate,min_price_per_event,max_price_per_event,pricing_type,rating,availability,event_type,review_count,is_synthetic
0,VND-001,Synthetic Catering Vendor 001,Catering,Vashi,NaN,NaN,37051.0,89468.0,per_event,4.7,Unavailable,Birthday|Anniversary,422,True
1,VND-002,Synthetic Photography Vendor 002,Photography,Vashi,NaN,NaN,29373.0,56879.0,per_event,3.3,Available,Wedding,343,True
2,VND-003,Synthetic Decoration Vendor 003,Decoration,CBD Belapur,NaN,NaN,67318.0,67411.0,per_event,4.6,Available,Anniversary|Baby Shower|Corporate Event,255,True
3,VND-004,Synthetic Event Planning Vendor 004,Event Planning,CBD Belapur,NaN,NaN,76874.0,78481.0,per_event,4.3,Available,Wedding,61,True
4,VND-005,Synthetic Makeup Artist Vendor 005,Makeup Artist,Vashi,NaN,NaN,21811.0,23614.0,per_event,4.3,Available,Corporate Event,49,True


In [24]:
# Check dataset dimensions
print("Dataset shape:", df.shape)

# Check column names
print("\nColumn names:")
print(df.columns.tolist())

# Check data types and non-null counts
print("\nDataset information:")
df.info()

Dataset shape: (60, 14)

Column names:
['vendor_id', 'vendor_name', 'category', 'location', 'min_price_per_plate', 'max_price_per_plate', 'min_price_per_event', 'max_price_per_event', 'pricing_type', 'rating', 'availability', 'event_type', 'review_count', 'is_synthetic']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   vendor_id            60 non-null     str    
 1   vendor_name          60 non-null     str    
 2   category             60 non-null     str    
 3   location             60 non-null     str    
 4   min_price_per_plate  4 non-null      float64
 5   max_price_per_plate  4 non-null      float64
 6   min_price_per_event  58 non-null     float64
 7   max_price_per_event  58 non-null     float64
 8   pricing_type         60 non-null     str    
 9   rating               60 non-null     float64
 10  availabil

In [25]:
# Count missing values in each column
print("Missing values:")
print(df.isnull().sum())

Missing values:
vendor_id               0
vendor_name             0
category                0
location                0
min_price_per_plate    56
max_price_per_plate    56
min_price_per_event     2
max_price_per_event     2
pricing_type            0
rating                  0
availability            0
event_type              0
review_count            0
is_synthetic            0
dtype: int64


In [26]:
# Check duplicate vendor IDs
print("Duplicate vendor IDs:", df["vendor_id"].duplicated().sum())

# Check completely duplicated rows
print("Duplicate rows:", df.duplicated().sum())

Duplicate vendor IDs: 0
Duplicate rows: 0


In [27]:
print("Vendor categories:")
print(df["category"].value_counts())

print("\nVendor locations:")
print(df["location"].value_counts())

print("\nPricing types:")
print(df["pricing_type"].value_counts())

Vendor categories:
category
Catering          10
Photography       10
Decoration        10
Event Planning    10
Makeup Artist     10
Entertainment     10
Name: count, dtype: int64

Vendor locations:
location
Vashi          20
CBD Belapur    20
Nerul          20
Name: count, dtype: int64

Pricing types:
pricing_type
per_event    56
per_plate     2
both          2
Name: count, dtype: int64


In [28]:
# Check whether vendor IDs are unique
print("Unique vendor IDs:", df["vendor_id"].nunique())
print("Total vendor records:", len(df))

assert df["vendor_id"].is_unique

print("Vendor ID validation passed!")

Unique vendor IDs: 60
Total vendor records: 60
Vendor ID validation passed!


In [29]:
# Check rating range
print("Minimum rating:", df["rating"].min())
print("Maximum rating:", df["rating"].max())

invalid_ratings = df[
    (df["rating"] < 0) | (df["rating"] > 5)
]

print("Invalid ratings:", len(invalid_ratings))

Minimum rating: 3.0
Maximum rating: 5.0
Invalid ratings: 0


In [30]:
# Check for negative prices
price_columns = [
    "min_price_per_plate",
    "max_price_per_plate",
    "min_price_per_event",
    "max_price_per_event"
]

for column in price_columns:
    print(column, "negative values:", (df[column] < 0).sum())

min_price_per_plate negative values: 0
max_price_per_plate negative values: 0
min_price_per_event negative values: 0
max_price_per_event negative values: 0


In [31]:
# Check whether pricing_type matches available price fields

invalid_per_plate = df[
    (df["pricing_type"] == "per_plate") &
    (df["min_price_per_plate"].isna() |
     df["max_price_per_plate"].isna())
]

invalid_per_event = df[
    (df["pricing_type"] == "per_event") &
    (df["min_price_per_event"].isna() |
     df["max_price_per_event"].isna())
]

invalid_both = df[
    (df["pricing_type"] == "both") &
    (df["min_price_per_plate"].isna() |
     df["max_price_per_plate"].isna() |
     df["min_price_per_event"].isna() |
     df["max_price_per_event"].isna())
]

print("Invalid per-plate records:", len(invalid_per_plate))
print("Invalid per-event records:", len(invalid_per_event))
print("Invalid both records:", len(invalid_both))

Invalid per-plate records: 0
Invalid per-event records: 0
Invalid both records: 0


In [32]:
# Check missing values in the dataset

missing_values = df.isnull().sum()

print(missing_values)

vendor_id               0
vendor_name             0
category                0
location                0
min_price_per_plate    56
max_price_per_plate    56
min_price_per_event     2
max_price_per_event     2
pricing_type            0
rating                  0
availability            0
event_type              0
review_count            0
is_synthetic            0
dtype: int64


In [33]:
# Expected missing values based on pricing type

expected_missing = (
    ((df["pricing_type"] == "per_event") &
     df["min_price_per_plate"].isna() &
     df["max_price_per_plate"].isna())
    |
    ((df["pricing_type"] == "per_plate") &
     df["min_price_per_event"].isna() &
     df["max_price_per_event"].isna())
)

print("Records with expected missing pricing fields:",
      expected_missing.sum())

Records with expected missing pricing fields: 58


In [34]:
# Required fields should not be missing

required_columns = [
    "vendor_id",
    "vendor_name",
    "category",
    "location",
    "pricing_type",
    "rating",
    "availability",
    "event_type",
    "review_count",
    "is_synthetic"
]

print("Missing required values:")
print(df[required_columns].isnull().sum())

Missing required values:
vendor_id       0
vendor_name     0
category        0
location        0
pricing_type    0
rating          0
availability    0
event_type      0
review_count    0
is_synthetic    0
dtype: int64


In [35]:
# Count vendors by pricing type
print(df["pricing_type"].value_counts())

# Check missing prices for each pricing type
print("\nMissing pricing fields by pricing type:")

print(
    df.groupby("pricing_type")[
        [
            "min_price_per_plate",
            "max_price_per_plate",
            "min_price_per_event",
            "max_price_per_event"
        ]
    ].apply(lambda group: group.isnull().sum())
)

pricing_type
per_event    56
per_plate     2
both          2
Name: count, dtype: int64

Missing pricing fields by pricing type:
              min_price_per_plate  max_price_per_plate  min_price_per_event  \
pricing_type                                                                  
both                            0                    0                    0   
per_event                      56                   56                    0   
per_plate                       0                    0                    2   

              max_price_per_event  
pricing_type                       
both                            0  
per_event                       0  
per_plate                       2  


In [36]:
# Load the corrected dataset
df = pd.read_csv("../data/raw/vendors.csv")

print("Dataset shape:", df.shape)

print("\nPricing types:")
print(df["pricing_type"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (60, 14)

Pricing types:
pricing_type
per_event    56
per_plate     2
both          2
Name: count, dtype: int64

Missing values:
vendor_id               0
vendor_name             0
category                0
location                0
min_price_per_plate    56
max_price_per_plate    56
min_price_per_event     2
max_price_per_event     2
pricing_type            0
rating                  0
availability            0
event_type              0
review_count            0
is_synthetic            0
dtype: int64


In [37]:
# Create a separate copy for cleaning
clean_df = df.copy()

# Remove duplicate records, if any
clean_df = clean_df.drop_duplicates()

# Remove duplicate vendor IDs, if any
clean_df = clean_df.drop_duplicates(
    subset=["vendor_id"],
    keep="first"
)

# Remove extra spaces from text columns
text_columns = [
    "vendor_id",
    "vendor_name",
    "category",
    "location",
    "pricing_type",
    "availability",
    "event_type"
]

for column in text_columns:
    clean_df[column] = clean_df[column].str.strip()

# Convert numeric columns to appropriate types
numeric_columns = [
    "min_price_per_plate",
    "max_price_per_plate",
    "min_price_per_event",
    "max_price_per_event",
    "rating",
    "review_count"
]

for column in numeric_columns:
    clean_df[column] = pd.to_numeric(
        clean_df[column],
        errors="coerce"
    )

print("Cleaning completed!")
print("Original records:", len(df))
print("Cleaned records:", len(clean_df))
print("Cleaned shape:", clean_df.shape)

Cleaning completed!
Original records: 60
Cleaned records: 60
Cleaned shape: (60, 14)


In [38]:
print("Cleaned dataset shape:", clean_df.shape)

print(
    "Duplicate vendor IDs:",
    clean_df["vendor_id"].duplicated().sum()
)

print(
    "Duplicate rows:",
    clean_df.duplicated().sum()
)

Cleaned dataset shape: (60, 14)
Duplicate vendor IDs: 0
Duplicate rows: 0


In [39]:
print("Raw dataset rating distribution:")
print(df["rating"].value_counts().sort_index())

print("\nCleaned dataset rating distribution:")
print(clean_df["rating"].value_counts().sort_index())

print("\nNumber of unique ratings:", clean_df["rating"].nunique())

Raw dataset rating distribution:
rating
3.0    1
3.1    2
3.2    5
3.3    3
3.4    2
3.5    2
3.6    2
3.7    3
3.8    2
3.9    5
4.0    1
4.1    1
4.2    1
4.3    5
4.4    2
4.5    4
4.6    6
4.7    3
4.8    5
4.9    4
5.0    1
Name: count, dtype: int64

Cleaned dataset rating distribution:
rating
3.0    1
3.1    2
3.2    5
3.3    3
3.4    2
3.5    2
3.6    2
3.7    3
3.8    2
3.9    5
4.0    1
4.1    1
4.2    1
4.3    5
4.4    2
4.5    4
4.6    6
4.7    3
4.8    5
4.9    4
5.0    1
Name: count, dtype: int64

Number of unique ratings: 21


In [40]:
print("Minimum rating:", clean_df["rating"].min())
print("Maximum rating:", clean_df["rating"].max())
print("Unique ratings:", clean_df["rating"].nunique())


Minimum rating: 3.0
Maximum rating: 5.0
Unique ratings: 21


In [41]:
# Save the cleaned dataset
clean_df.to_csv(
    "../data/clean/vendors_clean.csv",
    index=False
)

print("Cleaned dataset saved successfully!")
print("Final shape:", clean_df.shape)

Cleaned dataset saved successfully!
Final shape: (60, 14)
